# SSL Autoencoder Embeddings + t-SNE

Train an SSL autoencoder using the MARVEL dual-branch encoder (`ResNet` for MFCC, `EfficientNet` for log-mel), on **merged train + test** data using only audio features (`mfcc`, `mel_spectrogram`).

Pipeline:
1. Load cleaned train/test parquet
2. Keep audio tensors + metadata
3. Train `MarvelSSLAutoencoder` for reconstruction
4. Extract + normalize embeddings
5. Visualize t-SNE:
   - colored by `diagnosis_parkinsons_ds`
   - colored by `voice_perception__voice_quality_perception`
   - longitudinal evolution lines for test patients


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE

from marvel_model import MarvelSSLAutoencoder

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DATA_DIR = Path('/home/srl/B2AI/LLM/data')
TRAIN_PATH = DATA_DIR / 'cleaned_non_longitudinal_parkinson.parquet'
TEST_PATH = DATA_DIR / 'cleaned_longitudinal_parkinson.parquet'

print('Train path:', TRAIN_PATH)
print('Test  path:', TEST_PATH)
print('Train exists:', TRAIN_PATH.exists())
print('Test  exists:', TEST_PATH.exists())


In [ ]:
def to_2d_float32(x):
    arr = np.array(x, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D array, got shape {arr.shape}')
    return arr


def pad_or_truncate_2d(mat, target_t):
    # mat shape: (C, T)
    t = mat.shape[1]
    if t == target_t:
        return mat
    if t > target_t:
        return mat[:, :target_t]
    pad = np.zeros((mat.shape[0], target_t - t), dtype=mat.dtype)
    return np.concatenate([mat, pad], axis=1)


# Fixed sizes for CNN input
MFCC_T = 512
MEL_T = 512

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

train_df = train_df.copy()
test_df = test_df.copy()
train_df['split'] = 'train'
test_df['split'] = 'test'

all_df = pd.concat([train_df, test_df], ignore_index=True)
print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('All shape  :', all_df.shape)

# Keep only audio + metadata for analysis
meta_cols = ['participant_id', 'session_id', 'split']
for c in ['task_name', 'diagnosis_parkinsons_ds', 'voice_perception__voice_quality_perception', 'session_order']:
    if c in all_df.columns:
        meta_cols.append(c)

meta_df = all_df[meta_cols].copy()

mfcc_list = []
mel_list = []
for _, row in all_df.iterrows():
    mfcc = to_2d_float32(row['mfcc'])
    mel = to_2d_float32(row['mel_spectrogram'])

    # fixed T for batching
    mfcc = pad_or_truncate_2d(mfcc, MFCC_T)
    mel = pad_or_truncate_2d(mel, MEL_T)

    # per-recording z-norm (robust training)
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)
    mel = (mel - mel.mean(axis=1, keepdims=True)) / (mel.std(axis=1, keepdims=True) + 1e-8)

    mfcc_list.append(mfcc)
    mel_list.append(mel)

X_mfcc = np.stack(mfcc_list).astype(np.float32)  # (N, C_mfcc, T)
X_mel = np.stack(mel_list).astype(np.float32)    # (N, C_mel, T)

print('MFCC tensor:', X_mfcc.shape)
print('Log-mel tensor:', X_mel.shape)


In [ ]:
class AudioSSLDataset(Dataset):
    def __init__(self, x_mfcc, x_mel):
        # convert to (N, 1, H, W)
        self.x_mfcc = torch.tensor(x_mfcc[:, None, :, :], dtype=torch.float32)
        self.x_mel = torch.tensor(x_mel[:, None, :, :], dtype=torch.float32)

    def __len__(self):
        return self.x_mfcc.shape[0]

    def __getitem__(self, idx):
        return self.x_mfcc[idx], self.x_mel[idx]


def add_input_noise(x, std=0.02):
    if std <= 0:
        return x
    return x + std * torch.randn_like(x)


dataset = AudioSSLDataset(X_mfcc, X_mel)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=False)

model = MarvelSSLAutoencoder(shared_dim=512, pretrained=False).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 20
loss_hist = []

model.train()
for epoch in range(1, epochs + 1):
    running = 0.0
    n_batches = 0

    for xb_mfcc, xb_mel in loader:
        xb_mfcc = xb_mfcc.to(device)
        xb_mel = xb_mel.to(device)

        # denoising autoencoder flavor
        in_mfcc = add_input_noise(xb_mfcc, std=0.02)
        in_mel = add_input_noise(xb_mel, std=0.02)

        recon_mfcc, recon_mel = model(in_mfcc, in_mel)
        loss = model.reconstruction_loss(
            x_mfcc=xb_mfcc,
            x_spec=xb_mel,
            recon_mfcc=recon_mfcc,
            recon_spec=recon_mel,
            mfcc_weight=1.0,
            spec_weight=1.0,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item()
        n_batches += 1

    epoch_loss = running / max(n_batches, 1)
    loss_hist.append(epoch_loss)
    print(f'Epoch {epoch:03d} | Recon loss: {epoch_loss:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(loss_hist)
plt.title('SSL Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction loss')
plt.grid(alpha=0.2)
plt.show()


In [ ]:
# Extract embeddings for all recordings
model.eval()
with torch.no_grad():
    x_mfcc_t = torch.tensor(X_mfcc[:, None, :, :], dtype=torch.float32, device=device)
    x_mel_t = torch.tensor(X_mel[:, None, :, :], dtype=torch.float32, device=device)
    _, _, z = model(x_mfcc_t, x_mel_t, return_embedding=True)
    emb = z.cpu().numpy().astype(np.float32)

# L2-normalize embeddings
emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
print('Embeddings shape:', emb.shape)

emb_df = meta_df.copy()
for j in range(emb.shape[1]):
    emb_df[f'emb_{j:03d}'] = emb[:, j]

# Save embeddings with metadata
out_path = DATA_DIR / 'ssl_embeddings_marvel_autoencoder.parquet'
emb_df.to_parquet(out_path, index=False)
print('Saved:', out_path)
print('Saved shape:', emb_df.shape)


In [ ]:
# t-SNE projection on embeddings
print('Running t-SNE...')
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=SEED,
)
emb_2d = tsne.fit_transform(emb)

plot_df = emb_df[['participant_id', 'session_id', 'split']].copy()
for c in ['task_name', 'diagnosis_parkinsons_ds', 'voice_perception__voice_quality_perception', 'session_order']:
    if c in emb_df.columns:
        plot_df[c] = emb_df[c]
plot_df['tsne_x'] = emb_2d[:, 0]
plot_df['tsne_y'] = emb_2d[:, 1]

# 1) Split plot
plt.figure(figsize=(8, 6))
for split, color in [('train', 'tab:blue'), ('test', 'tab:orange')]:
    sub = plot_df[plot_df['split'] == split]
    plt.scatter(sub['tsne_x'], sub['tsne_y'], s=8, alpha=0.6, c=color, label=split)
plt.title('t-SNE embeddings (train + test)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# 2) Color by diagnosis_parkinsons_ds
if 'diagnosis_parkinsons_ds' in plot_df.columns:
    sev = pd.to_numeric(plot_df['diagnosis_parkinsons_ds'], errors='coerce')
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(plot_df['tsne_x'], plot_df['tsne_y'], c=sev, s=8, alpha=0.7, cmap='plasma')
    plt.colorbar(sc, label='diagnosis_parkinsons_ds')
    plt.title('t-SNE colored by raw H&Y severity')
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.grid(alpha=0.2)
    plt.show()

# 3) Color by voice perception quality
if 'voice_perception__voice_quality_perception' in plot_df.columns:
    vq = pd.to_numeric(plot_df['voice_perception__voice_quality_perception'], errors='coerce')
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(plot_df['tsne_x'], plot_df['tsne_y'], c=vq, s=8, alpha=0.7, cmap='viridis')
    plt.colorbar(sc, label='voice_perception__voice_quality_perception')
    plt.title('t-SNE colored by voice quality perception')
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.grid(alpha=0.2)
    plt.show()


In [ ]:
# Longitudinal evolution lines (TEST only), colored by voice quality

test_df_plot = plot_df[plot_df['split'] == 'test'].copy()

# Aggregate recording-level embeddings into session-level centroids for clean trajectories
agg_cols = {
    'tsne_x': 'mean',
    'tsne_y': 'mean',
}
if 'voice_perception__voice_quality_perception' in test_df_plot.columns:
    agg_cols['voice_perception__voice_quality_perception'] = 'mean'
if 'diagnosis_parkinsons_ds' in test_df_plot.columns:
    agg_cols['diagnosis_parkinsons_ds'] = 'mean'
if 'session_order' in test_df_plot.columns:
    agg_cols['session_order'] = 'mean'

session_centroids = (
    test_df_plot
    .groupby(['participant_id', 'session_id'], as_index=False)
    .agg(agg_cols)
)

if 'session_order' not in session_centroids.columns:
    # fallback if session_order unavailable
    session_centroids = session_centroids.sort_values(['participant_id', 'session_id'])
    session_centroids['session_order'] = (
        session_centroids.groupby('participant_id').cumcount() + 1
    )

session_centroids = session_centroids.sort_values(['participant_id', 'session_order'])

plt.figure(figsize=(9, 7))

# Background all test session centroids
plt.scatter(session_centroids['tsne_x'], session_centroids['tsne_y'], s=25, alpha=0.3, c='gray')

cmap = plt.cm.viridis
norm_vals = None
if 'voice_perception__voice_quality_perception' in session_centroids.columns:
    norm_vals = pd.to_numeric(
        session_centroids['voice_perception__voice_quality_perception'], errors='coerce'
    )

for pid, grp in session_centroids.groupby('participant_id'):
    grp = grp.sort_values('session_order')
    if len(grp) < 2:
        continue

    pts = grp[['tsne_x', 'tsne_y']].to_numpy()
    segs = np.stack([pts[:-1], pts[1:]], axis=1)

    if norm_vals is not None:
        c = pd.to_numeric(grp['voice_perception__voice_quality_perception'], errors='coerce').to_numpy()
        c_line = c[1:] if len(c) > 1 else np.array([np.nan])
        c_line = np.nan_to_num(c_line, nan=np.nanmean(norm_vals.to_numpy(dtype=float)))
        lc = LineCollection(segs, cmap=cmap, linewidths=2.0, alpha=0.95)
        lc.set_array(c_line)
        plt.gca().add_collection(lc)
    else:
        plt.plot(grp['tsne_x'], grp['tsne_y'], '-', linewidth=2.0, alpha=0.8)

    # Mark start/end
    plt.scatter(grp['tsne_x'].iloc[0], grp['tsne_y'].iloc[0], marker='o', s=45, c='black', alpha=0.85)
    plt.scatter(grp['tsne_x'].iloc[-1], grp['tsne_y'].iloc[-1], marker='X', s=55, c='red', alpha=0.85)

if norm_vals is not None:
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_array(pd.to_numeric(session_centroids['voice_perception__voice_quality_perception'], errors='coerce').dropna())
    plt.colorbar(sm, label='voice_perception__voice_quality_perception')

plt.title('Longitudinal trajectories on t-SNE (test patients)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.grid(alpha=0.2)
plt.show()

print('Session-centroid rows:', len(session_centroids))
print('Patients in trajectories:', session_centroids['participant_id'].nunique())
